In [ ]:
# Data Cleaning
# Keeping only the relevant columns and checking for invalid or missing data.

import pandas as pd

income = pd.read_csv("../dataset/raw/income.csv")
balance = pd.read_csv("../dataset/raw/balance.csv") 
cashflow = pd.read_csv("../dataset/raw/cashflow.csv")
prices = pd.read_csv("../dataset/raw/prices.csv")
companies = pd.read_csv("../dataset/raw/companies.csv")

In [ ]:
# Checking the columns of the income statement
print(income.columns.tolist())

# Selecting relevant columns from the income statement
income_cols = ["Ticker", "Fiscal Year", "Report Date", "Revenue",
               "Operating Income (Loss)", "Interest Expense, Net",
               "Net Income", "Shares (Basic)"]
income = income[income_cols]

# Checking for missing values, gaps, and invalid data in the income statement
gaps = income.sort_values(["Ticker", "Fiscal Year"]).groupby("Ticker")["Fiscal Year"].diff()
problem = (
    (gaps > 1)
    | income.isna().any(axis=1)
    | (income["Revenue"] <= 0)
    | (income["Shares (Basic)"] <= 0)
)
print(income[problem])

# Fixing missing values in the income statement
income = income.fillna(0)

# After filling missing values:
print(income[income.isna().any(axis=1)])

['Ticker', 'Report Date', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Expenses', 'Selling, General & Administrative', 'Research & Development', 'Depreciation & Amortization', 'Operating Income (Loss)', 'Non-Operating Income (Loss)', 'Interest Expense, Net', 'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)', 'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net', 'Income (Loss) from Continuing Operations', 'Net Extraordinary Gains (Losses)', 'Net Income', 'Net Income (Common)']
   Ticker  Fiscal Year Report Date       Revenue  Operating Income (Loss)  \
40   INTC         2020  2020-12-31  7.786700e+10             2.387600e+10   
41   INTC         2021  2021-12-31  7.902400e+10             2.208200e+10   
42   INTC         2022  2022-12-31  6.305400e+10             2.336000e+09   
43   INTC         2023  2023-12-31  5.422800e+10           

In [ ]:
# Checking the columns of the balance sheet
print(balance.columns.tolist())

# Selecting relevant columns from the balance sheet
balance_cols = ["Ticker", "Fiscal Year", "Report Date",
                "Cash, Cash Equivalents & Short Term Investments", "Inventories",
                "Total Current Assets", "Total Assets",
                "Short Term Debt", "Total Current Liabilities", "Long Term Debt",
                "Total Liabilities", "Retained Earnings", "Total Equity",
                "Total Liabilities & Equity"]
balance = balance[balance_cols]

# Checking for missing values, gaps, and invalid data in the balance sheet
balance = balance.sort_values(["Ticker", "Fiscal Year"])
gaps = balance.groupby("Ticker")["Fiscal Year"].diff()

core = ["Total Assets", "Total Current Assets", "Total Current Liabilities",
        "Total Liabilities", "Total Equity", "Retained Earnings",
        "Cash, Cash Equivalents & Short Term Investments"]

identity_diff = (balance["Total Assets"] - balance["Total Liabilities"] - balance["Total Equity"]).abs()

problem = (
    (gaps > 1)
    | balance[core].isna().any(axis=1)
    | (balance["Total Assets"] <= 0)
    | (balance["Total Current Liabilities"] <= 0)
    | (identity_diff / balance["Total Assets"] > 0.01)
    | balance.duplicated(["Ticker", "Fiscal Year"], keep=False)
)
print(balance[problem])

# Negative equity does not break the data, but it distorts ROE and debt/equity
print(balance[balance["Total Equity"] <= 0][["Ticker", "Fiscal Year", "Total Equity"]])
# There are companies with negative equity, so we will keep attention to them when analyzing the data.

# Fixing missing values: 0 only where NaN means "none"
zero_cols = ["Short Term Debt", "Long Term Debt", "Inventories"]
balance[zero_cols] = balance[zero_cols].fillna(0)
balance = balance.dropna(subset=core)

# After fixing:
print(balance[balance.isna().any(axis=1)])

['Ticker', 'Report Date', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Other Long Term Assets', 'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals', 'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt', 'Total Noncurrent Liabilities', 'Total Liabilities', 'Share Capital & Additional Paid-In Capital', 'Treasury Stock', 'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity']
Empty DataFrame
Columns: [Ticker, Fiscal Year, Report Date, Cash, Cash Equivalents & Short Term Investments, Inventories, Total Current Assets, Total Assets, Short Term Debt, Total Current Liabilities, Long Term Debt, Total Liabilities, Retained Earnings, Total Equity, Total Liabilities & Equity]
Index: []
 